# HubbardBath 10 — Explicit QSP phase synthesis

This notebook takes the bounded thermal polynomial from Milestone 08 and turns its even and odd Chebyshev components into **actual symmetric-QSP phase sequences**.

A single standard QSP sequence has definite parity, so the full thermal polynomial is handled as

$$p(x)=p_{\rm even}(x)+p_{\rm odd}(x).$$

Each component is synthesized separately, then its signal response is reconstructed directly from the returned phase sequence.

In [ ]:
from pathlib import Path
import sys
import numpy as np
import matplotlib.pyplot as plt
from numpy.polynomial.chebyshev import chebval

candidates=[Path.cwd(),Path.cwd().parent,Path.cwd()/'HubbardBath']
for c in candidates:
    if (c/'src'/'qsp_phases.py').exists():
        repo_root=c.resolve(); break
else:
    raise FileNotFoundError('Run from the HubbardBath repository')
if str(repo_root) not in sys.path:
    sys.path.insert(0,str(repo_root))

from src.hubbard import two_site_hubbard_hamiltonian
from src.qsp_phases import (
    qsp_imaginary_response,
    reconstructed_thermal_response,
    synthesize_thermal_parity_components,
    thermal_qsp_reconstruction_error,
)
from src.qubit_mapping import lcu_normalization, remove_zero_terms, two_site_hubbard_pauli_terms
from src.thermal_polynomial import minimum_degree_for_sector_gibbs_error


## 1. Rebuild the reference thermal polynomial

We use the same reference point $U/t=4$, $\beta t=1$ and choose the minimum bounded polynomial whose half-filled Gibbs state reaches trace distance $10^{-2}$.

In [ ]:
H=two_site_hubbard_hamiltonian(t=1.0,U=4.0,mu=0.0)
terms=remove_zero_terms(two_site_hubbard_pauli_terms(t=1.0,U=4.0,mu=0.0))
alpha=lcu_normalization(terms)
approx,gibbs_error=minimum_degree_for_sector_gibbs_error(
    H,beta=1.0,alpha=alpha,target_trace_distance=1e-2,max_degree=64
)
print('alpha:',alpha)
print('thermal polynomial degree:',approx.degree)
print('achieved Gibbs trace distance:',gibbs_error)


## 2. Synthesize the parity components

The symmetric-QSP phase finder receives parity-reduced Chebyshev coefficients. The achieved polynomial appears as the imaginary part of the top-left matrix element in the convention used here.

In [ ]:
even,odd=synthesize_thermal_parity_components(approx,crit=1e-12,grid_size=4001)
print('Even degree / phases / error:',even.polynomial_degree,len(even.full_phases),even.max_response_error)
print('Odd degree / phases / error: ',odd.polynomial_degree,len(odd.full_phases),odd.max_response_error)
print('Total reconstruction error:',thermal_qsp_reconstruction_error(approx,even,odd))


## 3. Inspect the actual phase angles

These are explicit phase sequences, not placeholder parameters.

In [ ]:
print('Even phases:')
for i,phi in enumerate(even.full_phases):
    print(f'{i:2d}: {phi:+.12f}')
print('\nOdd phases:')
for i,phi in enumerate(odd.full_phases):
    print(f'{i:2d}: {phi:+.12f}')


## 4. Independently reconstruct the signal response

HubbardBath evaluates the QSP matrix product directly from the returned phases and compares it with the original Chebyshev polynomial on a dense grid.

In [ ]:
x=np.linspace(-1,1,1001)
target=chebval(x,approx.coefficients)
even_response=qsp_imaginary_response(x,even.full_phases)
odd_response=qsp_imaginary_response(x,odd.full_phases)
total=reconstructed_thermal_response(x,even,odd)

plt.figure(figsize=(7,4.8))
plt.plot(x,target,label='thermal polynomial')
plt.plot(x,total,linestyle='--',label='synthesized QSP sum')
plt.xlabel(r'$x$'); plt.ylabel('signal response')
plt.title('Explicit QSP reconstruction')
plt.legend(); plt.grid(alpha=0.2); plt.tight_layout(); plt.show()


## What we can claim now

We have **explicit symmetric-QSP phase sequences** for the even and odd components and have reconstructed their signal responses numerically.

We still do not call this a compiled fault-tolerant QSVT Gibbs-preparation circuit: coherent combination of the two parity sequences and hardware-level synthesis remain separate steps.